# 9장. 모델 선택과 검증 자료

여러 후보 가운데 골라낸 모델의 지표는 그 모델의 실제 성능인가.

**준비:** 1장에서 쓴 한국전력거래소 「시간별 전국 전력수요량」 CSV 세 개와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개, 6장에서 쓴 경기도 「일반음식점(중국식) 현황」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 도는 데 이삼 분이 걸린다.

## 9.1 시험 자료로 하이퍼파라미터를 고를 때의 편향

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

def neighbors(k):
    """기온 t와 수요 y를 받아, 기온이 가장 가까운 k일의 수요를 평균해 예측하는 함수를 만든다."""
    def make(t, y):
        return lambda q: np.array([y[np.argsort(np.abs(t - v))[:k]].mean() for v in q])
    return make

def polynomial(degree):
    """기온 t와 수요 y에 degree차 다항식을 추정해, 기온을 넣으면 수요를 예측하는 함수를 만든다."""
    def make(t, y):
        mu, sd = t.mean(), t.std()
        coef = np.linalg.lstsq(np.vander((t - mu) / sd, degree + 1), y, rcond=None)[0]
        return lambda q: np.vander((q - mu) / sd, degree + 1) @ coef
    return make

KS = [1, 2, 3, 5, 8, 12, 20, 30, 50, 100, 247]
# 2023년으로 만든 모델의 2025년 예측
answers = {k: neighbors(k)(T23, Y23)(T25) for k in KS}
peek = pd.Series({k: rmse(Y25, answers[k]) for k in KS})
print(peek.round(2).to_string())
print(int(peek.idxmin()), round(float(peek.min()), 2))

In [ ]:
def folds(n, rng, parts=5):
    return np.array_split(rng.permutation(n), parts)

def crossval(make, t, y, parts):
    """묶음을 하나씩 빼고 나머지로 만든 모델로 뺀 묶음을 예측한다. 시험 자료는 쓰지 않는다."""
    sq = []
    for held in parts:
        seen = np.ones(len(t), dtype=bool)
        seen[held] = False
        sq.append((y[held] - make(t[seen], y[seen])(t[held])) ** 2)
    return float(np.sqrt(np.concatenate(sq).mean()))

parts = folds(len(Y23), np.random.default_rng(0))
within = pd.Series({k: crossval(neighbors(k), T23, Y23, parts) for k in KS})
print(within.round(2).to_string())
k_in = int(within.idxmin())
print(k_in, round(float(within.min()), 2), round(float(peek[k_in]), 2))
print([int(pd.Series({k: crossval(neighbors(k), T23, Y23, folds(len(Y23), np.random.default_rng(s))) for k in KS}).idxmin()) for s in range(1, 8)])

In [ ]:
rng = np.random.default_rng(0)
n25 = len(Y25)
sq = {k: (Y25 - answers[k]) ** 2 for k in KS}

picked, seen_half, other_half = [], [], []
for _ in range(2000):
    order = rng.permutation(n25)
    # 2025년을 무작위로 절반씩 나눈다
    a, b = order[: n25 // 2], order[n25 // 2:]
    # 한쪽 절반의 오차로 이웃 수를 고른다
    k = min(KS, key=lambda k: sq[k][a].mean())
    picked.append(k)
    seen_half.append(np.sqrt(sq[k][a].mean()))
    other_half.append(np.sqrt(sq[k][b].mean()))
print(pd.Series(picked).value_counts().to_dict())
print(round(float(np.mean(seen_half)), 2), round(float(np.mean(other_half)), 2))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# 그림 9-1
shown_k = [k for k in KS if k <= 50]
x = np.arange(len(shown_k))
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(x, peek[shown_k], color=BRICK, lw=1.8, marker="o", ms=5, label="2025년 시험 자료의 RMSE")
ax.plot(x, within[shown_k], color=NAVY, lw=1.8, marker="s", ms=5, label="2023년 안에서의 5겹 교차검증 RMSE")
i12, i20 = shown_k.index(12), shown_k.index(20)
ax.annotate(f"시험 자료로 고르면 k = 12\n({peek[12]:.2f}GW)", (i12, peek[12]), (i12 - 2.6, 3.95), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.annotate(f"교차검증으로 고르면 k = 20\n({within[20]:.2f}GW)", (i20, within[20]), (i20 - 1.2, 2.62), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.scatter([i20], [peek[20]], s=70, facecolor="none", edgecolor=TEAL, lw=1.6, zorder=4)
ax.annotate(f"k = 20의 시험 RMSE {peek[20]:.2f}GW", (i20, peek[20]), (i20 + 0.25, 3.75), fontsize=8, color=TEAL, arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
ax.set_xticks(x, [str(k) for k in shown_k])
ax.set_ylim(2.5, 4.5)
ax.set_xlabel("이웃의 수 k", fontsize=8)
ax.set_ylabel("RMSE(GW)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_9_1.png", dpi=300)
plt.show()

## 9.2 후보가 많을 때: 검증 자료의 크기와 선택에 따른 편향

In [ ]:
name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

X, Xt, ytr = columns(train), columns(test), train["폐업함"].to_numpy()
score = Xt @ np.linalg.lstsq(X, ytr, rcond=None)[0]
print(X.shape, round(auc(y, score), 3))

In [ ]:
def auc_each(actual, scores):
    """scores의 열마다 AUC를 구한다. 열 하나가 후보 하나다."""
    rank = pd.DataFrame(scores).rank().to_numpy()
    n1 = actual.sum()
    return (rank[actual == 1].sum(axis=0) - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1))

rng = np.random.default_rng(0)
n = len(y)
dice = rng.random((n, 1000))                         # 가게마다 무작위 숫자를 매긴 점수 1,000벌
whole = auc_each(y, dice)
print(round(float(whole.mean()), 3), round(float(whole.std()), 4), round(float(whole.max()), 3))

small = rng.choice(n, 400, replace=False)            # 시험 자료가 400곳뿐이라면
rest = np.setdiff1d(np.arange(n), small)
few = auc_each(y[small], dice[small])
lucky = int(few.argmax())
print(round(float(few.std()), 4), round(float(few.max()), 3), round(auc(y[rest], dice[rest, lucky]), 3))
print([round(float(np.mean([few[rng.choice(1000, c, replace=False)].max() for _ in range(200)])), 3) for c in [1, 10, 100, 1000]])
z = np.random.default_rng(1).standard_normal((2000, 1000))
print([round(float(z[:, :c].max(axis=1).mean()), 2) for c in [1, 10, 100, 1000]], [round(float(np.sqrt(2 * np.log(c))), 2) for c in [10, 100, 1000]])

In [ ]:
# 그림 9-2
bins = np.arange(0.36, 0.661, 0.005)
fig, ax = plt.subplots(figsize=(7, 4.0))
ax.hist(few, bins=bins, color=to_rgba(BRICK, 0.45), edgecolor=BRICK, lw=0.5, label="시험 자료 400곳에서 구한 AUC")
ax.hist(whole, bins=bins, color=to_rgba(NAVY, 0.3), edgecolor=NAVY, lw=0.5, label="시험 자료 6,311곳에서 구한 AUC")
top = ax.get_ylim()[1]
ax.set_ylim(0, top * 1.18)
ax.axvline(auc(y, score), color=TEAL, lw=1.4)
ax.text(auc(y, score) + 0.003, top * 1.08, f"6장의 모델 {auc(y, score):.3f}", fontsize=8, color=TEAL)
ax.annotate(f"400곳의 최댓값 {few.max():.3f}", (few.max(), 1), (0.607, top * 0.34), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.annotate(f"6,311곳의 최댓값 {whole.max():.3f}", (whole.max(), 2), (0.607, top * 0.72), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.set_xlabel("무작위 점수 1,000벌의 AUC", fontsize=8)
ax.set_ylabel("후보의 수", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_9_2.png", dpi=300)
plt.show()

In [ ]:
year = train["기준일"].dt.year.to_numpy()
fit, tune = year < 2020, year == 2020               # 추정에 쓰는 줄, 선택에 쓰는 줄(검증 자료). 시험 자료는 2023년 기준일이다
yv = ytr[tune]

N = 1000
V, S = np.zeros((tune.sum(), N)), np.zeros((n, N))
for i in range(N):
    cols = np.where(rng.random(41) < 0.5)[0]         # 특성 41개를 각각 절반의 확률로 넣거나 뺀다
    coef = np.linalg.lstsq(X[fit][:, cols], ytr[fit], rcond=None)[0]
    V[:, i], S[:, i] = X[tune][:, cols] @ coef, Xt[:, cols] @ coef
on_test = auc_each(y, S)
on_val = auc_each(yv, V)
print(round(float(on_test.mean()), 3), round(float(on_test.std()), 4), round(float(on_test.max()), 3), round(float(np.corrcoef(on_val, on_test)[0, 1]), 2))

rows = []
for size in [len(yv), 1300, 330]:
    for count in [1, 10, 100, 1000]:
        marks = []
        for _ in range(20):
            seen = rng.choice(len(yv), size, replace=False)        # 검증 자료에서 size곳만 쓴다
            cand = rng.choice(N, count, replace=False)             # 후보는 count개만 만든다
            val = auc_each(yv[seen], V[seen][:, cand])
            marks.append((val.max(), on_test[cand[val.argmax()]]))
        rows.append((size, count, *np.mean(marks, axis=0).round(3)))
pick = pd.DataFrame(rows, columns=["검증 가게 수", "후보 수", "고른 후보의 검증 AUC", "그 후보의 시험 AUC"])
for what in ["고른 후보의 검증 AUC", "그 후보의 시험 AUC"]:
    print(what)
    print(pick.pivot(index="검증 가게 수", columns="후보 수", values=what).sort_index(ascending=False).to_string())

In [ ]:
# 그림 9-4
fig, ax = plt.subplots(figsize=(7, 4.1))
look = {5259: NAVY, 1300: TEAL, 330: BRICK}
xs_ = np.arange(4)
for size, col in look.items():
    t = pick[pick["검증 가게 수"] == size]
    ax.plot(xs_, t["고른 후보의 검증 AUC"], color=col, lw=1.8, marker="o", ms=5, ls=(0, (4, 2)))
    ax.plot(xs_, t["그 후보의 시험 AUC"], color=col, lw=1.8, marker="s", ms=5)
    gap = t["고른 후보의 검증 AUC"].iloc[-1] - t["그 후보의 시험 AUC"].iloc[-1]
    ax.text(3.08, t["고른 후보의 검증 AUC"].iloc[-1], f"검증 {size:,}곳\n차이 {gap:+.3f}", fontsize=8, color=col, va="center", linespacing=1.4)
ax.plot([], [], color="#44505c", lw=1.8, marker="o", ms=5, ls=(0, (4, 2)), label="고른 후보의 검증 AUC")
ax.plot([], [], color="#44505c", lw=1.8, marker="s", ms=5, label="같은 후보의 시험 AUC")
ax.set_xticks(xs_, ["1", "10", "100", "1,000"])
ax.set_xlim(-0.2, 3.75); ax.set_ylim(0.51, 0.65)
ax.set_xlabel("만들어 본 후보의 수", fontsize=8)
ax.set_ylabel("AUC", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_9_4.png", dpi=300)
plt.show()

In [ ]:
# 그림 9-5
seen = np.random.default_rng(5).choice(len(yv), 330, replace=False)
val_small = auc_each(yv[seen], V[seen])
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.scatter(val_small, on_test, s=6, color=to_rgba(BRICK, 0.4), lw=0, label="가로축: 검증 자료 330곳에서의 AUC")
ax.scatter(on_val, on_test, s=6, color=to_rgba(NAVY, 0.5), lw=0, label="가로축: 검증 자료 5,259곳에서의 AUC")
for val, col, text, pos in [(val_small, BRICK, "330곳으로 고른 후보", (0.615, 0.497)), (on_val, NAVY, "5,259곳으로 고른 후보", (0.425, 0.607))]:
    k = int(val.argmax())
    ax.scatter([val[k]], [on_test[k]], s=70, facecolor="none", edgecolor=col, lw=1.8, zorder=4)
    ax.annotate(f"{text}\n검증 {val[k]:.3f}, 시험 {on_test[k]:.3f}", (val[k], on_test[k]), pos, fontsize=8, color=col, bbox=BOX, arrowprops={"arrowstyle": "->", "color": col, "lw": 0.8})
ax.plot([0.42, 0.68], [0.42, 0.68], color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.set_xlim(0.42, 0.68); ax.set_ylim(0.46, 0.625)
ax.set_xlabel("검증 자료에서의 AUC", fontsize=8)
ax.set_ylabel("시험 자료 6,311곳에서의 AUC", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="lower right", markerscale=3)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_9_5.png", dpi=300)
plt.show()

## 9.3 중첩 교차검증: 선택 과정까지 포함해서 평가하기

In [ ]:
CAND = {**{f"이웃 {k}일": neighbors(k) for k in [1, 2, 3, 5, 8, 12, 20]},
        **{f"{g}차 다항식": polynomial(g) for g in [1, 2, 3, 4, 5, 6]}}

def choose(t, y, rng):
    """후보 13개를 교차검증으로 평가해 RMSE가 가장 작은 후보의 이름과 그 RMSE를 돌려준다."""
    parts = folds(len(t), rng)
    marks = {name: crossval(make, t, y, parts) for name, make in CAND.items()}
    best = min(marks, key=marks.get)
    return best, marks[best]

rng = np.random.default_rng(1)
# 2023년 평일 가운데 30일만 가지고 있다고 가정한다
mine = rng.choice(len(Y23), 30, replace=False)
# 나머지 217일은 실제 오차를 확인할 때만 쓴다
others = np.setdiff1d(np.arange(len(Y23)), mine)
t30, y30 = T23[mine], Y23[mine]

best, mark = choose(t30, y30, rng)
truth = rmse(Y23[others], CAND[best](t30, y30)(T23[others]))
print(best, round(mark, 2), round(truth, 2))

In [ ]:
def nested(t, y, rng):
    """바깥 묶음을 하나 빼고, 남은 날들만으로 후보를 고르는 일부터 다시 한 뒤, 고른 후보로 뺀 묶음을 예측한다."""
    sq, names = [], []
    for held in folds(len(t), rng):
        seen = np.ones(len(t), dtype=bool)
        seen[held] = False
        # 뺀 묶음은 후보를 고르는 데에도 쓰지 않는다
        best, _ = choose(t[seen], y[seen], rng)
        names.append(best)
        sq.append((y[held] - CAND[best](t[seen], y[seen])(t[held])) ** 2)
    return float(np.sqrt(np.concatenate(sq).mean())), names

mark2, names = nested(t30, y30, rng)
print(round(mark2, 2), names)

In [ ]:
rows = []
for _ in range(200):
    mine = rng.choice(len(Y23), 30, replace=False)
    others = np.setdiff1d(np.arange(len(Y23)), mine)
    t30, y30 = T23[mine], Y23[mine]
    best, mark = choose(t30, y30, rng)
    rows.append((mark, nested(t30, y30, rng)[0], rmse(Y23[others], CAND[best](t30, y30)(T23[others]))))
trial = pd.DataFrame(rows, columns=["선택에 쓴 교차검증", "중첩 교차검증", "나머지 217일"])
print(trial.agg(["mean", "median"]).round(2).to_string())
print(round(float((trial["선택에 쓴 교차검증"] < trial["나머지 217일"]).mean()), 2), round(float((trial["중첩 교차검증"] < trial["나머지 217일"]).mean()), 2))

In [ ]:
# 그림 9-7
fig, ax = plt.subplots(figsize=(7, 3.8))
names_ = ["선택에 쓴 교차검증", "중첩 교차검증", "나머지 217일"]
labels_ = ["후보를 고를 때 쓴\n교차검증 RMSE", "중첩 교차검증\nRMSE", "나머지 217일에서 잰\n실제 RMSE"]
cols_ = [BRICK, NAVY, TEAL]
jit = np.random.default_rng(0).uniform(-0.16, 0.16, len(trial))
for i, (nm, col) in enumerate(zip(names_, cols_)):
    v = np.minimum(trial[nm].to_numpy(), 8.0)
    ax.scatter(v, np.full(len(v), 2 - i) + jit, s=7, color=to_rgba(col, 0.35), lw=0)
    med = float(trial[nm].median())
    ax.plot([med, med], [2 - i - 0.3, 2 - i + 0.3], color=col, lw=2.4)
    ax.text(med, 2 - i + 0.36, f"중앙값 {med:.2f}", fontsize=8, color=col, ha="center")
    over = int((trial[nm] > 8).sum())
    if over:
        ax.text(8.05, 2 - i - 0.33, f"8GW 초과 {over}번", fontsize=7.5, color=col, ha="right")
ax.set_yticks([2, 1, 0], labels_, fontsize=8)
ax.set_xlim(1.8, 8.1); ax.set_ylim(-0.6, 2.7)
ax.set_xlabel("RMSE(GW). 30일을 무작위로 뽑는 일을 200번 반복했고 점 하나가 한 번의 결과다. 8GW보다 큰 값은 8에 그렸다", fontsize=7.6)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_9_7.png", dpi=300)
plt.show()

In [ ]:
best, mark = choose(T23, Y23, rng)
print(best, round(mark, 2), round(nested(T23, Y23, rng)[0], 2), round(rmse(Y25, CAND[best](T23, Y23)(T25)), 2))

## 9.4 시험 자료를 반복해서 쓸 때의 과적합

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(n)
# 시험 자료의 절반은 공개하고 절반은 숨긴다
shown, hidden = order[: n // 2], order[n // 2:]

def marks(keep):
    cols = np.where(keep)[0]
    s = Xt[:, cols] @ np.linalg.lstsq(X[:, cols], ytr, rcond=None)[0]
    return auc(y[shown], s[shown]), auc(y[hidden], s[hidden])

keep = np.ones(41, dtype=bool)
now = marks(keep)
log = [now]
for _ in range(300):
    trial_keep = keep.copy()
    j = rng.integers(0, 41)
    # 특성 하나를 넣거나 뺀다
    trial_keep[j] = ~trial_keep[j]
    got = marks(trial_keep)
    # 공개한 절반의 AUC가 오르면 그 변경을 유지한다
    if got[0] > now[0]:
        keep, now = trial_keep, got
    log.append(now)
log = np.array(log)
print(pd.DataFrame(log[[0, 10, 30, 100, 300]], index=[0, 10, 30, 100, 300], columns=["공개한 절반", "숨긴 절반"]).round(3).to_string())
print(int(keep.sum()), int((np.diff(log[:, 0]) > 0).sum()))

In [ ]:
# 그림 9-8
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(log[:, 0], color=BRICK, lw=1.8, label="공개한 절반의 AUC(이 값이 오를 때만 변경을 유지했다)")
ax.plot(log[:, 1], color=NAVY, lw=1.8, label="숨긴 절반의 AUC")
ax.text(303, log[-1, 0], f"{log[-1, 0]:.3f}", fontsize=8, color=BRICK, va="center")
ax.text(303, log[-1, 1], f"{log[-1, 1]:.3f}", fontsize=8, color=NAVY, va="center")
ax.text(-3, log[0, 0] + 0.004, f"{log[0, 0]:.3f}", fontsize=8, color=BRICK, ha="right", va="center")
ax.text(-3, log[0, 1] - 0.004, f"{log[0, 1]:.3f}", fontsize=8, color=NAVY, ha="right", va="center")
ax.annotate("", (300, log[-1, 0]), (300, log[-1, 1]), arrowprops={"arrowstyle": "<->", "color": GRAY, "lw": 1.0})
ax.text(292, (log[-1, 0] + log[-1, 1]) / 2, f"차이 {log[-1, 0] - log[-1, 1]:.3f}", fontsize=8, color="#6b7480", ha="right", va="center")
ax.set_xlim(-28, 330); ax.set_ylim(0.58, 0.645)
ax.set_xlabel("특성을 넣거나 빼 본 횟수", fontsize=8)
ax.set_ylabel("AUC", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_9_8.png", dpi=300)
plt.show()